## Maths Concepts

### Concept 1: The Dot Product

#### Mathematical Explanation

The **dot product** (also called the inner product or scalar product) takes two vectors of equal length and returns a single real number (a scalar).

For two $n$-dimensional vectors:


$$\mathbf{u} = [u_1, u_2, \dots, u_n]$$

$$\mathbf{v} = [v_1, v_2, \dots, v_n]$$

**Algebraic Definition:**
Multiply matching elements together, then sum all the products:

$$\mathbf{u} \cdot \mathbf{v} = \sum_{i=1}^{n} u_i v_i = u_1 v_1 + u_2 v_2 + \dots + u_n v_n$$

**Geometric Definition:**
It relates directly to the lengths (magnitudes) of the vectors and the angle $\theta$ between them:

$$\mathbf{u} \cdot \mathbf{v} = \Vert{}\mathbf{u}\Vert{} \Vert{}\mathbf{v}\Vert{} \cos(\theta)$$

* If $\mathbf{u} \cdot \mathbf{v} > 0$: The vectors point generally in the same direction ($\theta < 90^\circ$).
* If $\mathbf{u} \cdot \mathbf{v} = 0$: The vectors are orthogonal (perpendicular, $\theta = 90^\circ$).
* If $\mathbf{u} \cdot \mathbf{v} < 0$: The vectors point generally in opposite directions ($\theta > 90^\circ$).

**Why it matters in AI:**

Every single linear layer or neuron calculates $z = \mathbf{w} \cdot \mathbf{x} + b$. The weights $\mathbf{w}$ act as a template, and the dot product measures how strongly the input $\mathbf{x}$ matches that template.

#### NumPy Implementation

In [3]:
import numpy as np  # noqa: EXE002

# 1. Define two 1D vectors of equal length
u = np.array([2.0, 3.0, 4.0, 1.9])
v = np.array([1.0, -1.0, 2.0, 0.5])

# --- Method A: Element-wise multiplication followed by sum ---
# u * v multiplies element-by-element: [2*1, 3*(-1), 4*2, 1.9*0.5] = [2, -3, 8, 0.95]
# np.sum adds them up: 2 + (-3) + 8 + 0.95 = 7.95
dot_manual = np.sum(u * v)

# --- Method B: Using NumPy's built-in dot function ---
dot_numpy = np.dot(u, v)

# --- Method C: Using Python's @ matrix/vector multiplication operator ---
dot_operator = u @ v

print(f"Vector u: {u}")
print(f"Vector v: {v}")
print(f"Manual Dot Product: {dot_manual}")
print(f"NumPy np.dot:       {dot_numpy}")
print(f"NumPy @ operator:   {dot_operator}")

Vector u: [2.  3.  4.  1.9]
Vector v: [ 1.  -1.   2.   0.5]
Manual Dot Product: 7.95
NumPy np.dot:       7.95
NumPy @ operator:   7.95


### Concept 2: Vector Projection as "Coordinate Measurement"

#### Mathematical Explanation

In standard Cartesian math, when you have a 2D point $(x, y)$, the number $x$ is just the projection of that point onto the standard horizontal axis $[1, 0]$.

**Projection answers one core question:**

> *"If the only direction that existed in the world was the line pointing along $\mathbf{v}$, where would $\mathbf{u}$ land, and how much of $\mathbf{u}$ would be captured?"*

Think of $\mathbf{v}$ as a custom coordinate axis or a custom "feature detector":

1. **Step 1: Find the Direction ($\hat{\mathbf{v}}$)**
Normalize $\mathbf{v}$ to have a length of 1 so its raw magnitude does not distort your measurement:

$$\hat{\mathbf{v}} = \frac{\mathbf{v}}{\Vert{}\mathbf{v}\Vert{}}$$


2. **Step 2: Measure Along That Direction (Scalar Projection)**
Take the dot product of $\mathbf{u}$ with this unit direction. This gives the exact coordinate of $\mathbf{u}$ along the $\mathbf{v}$-axis:

$$\text{scalar\_proj} = \mathbf{u} \cdot \hat{\mathbf{v}} = \frac{\mathbf{u} \cdot \mathbf{v}}{\Vert{}\mathbf{v}\Vert{}}$$


3. **Step 3: Reconstruct the Vector in Full Space (Vector Projection)**
Scale the unit direction $\hat{\mathbf{v}}$ by that coordinate length:

$$\text{proj}_{\mathbf{v}}(\mathbf{u}) = (\text{scalar\_proj}) \hat{\mathbf{v}} = \left(\frac{\mathbf{u} \cdot \mathbf{v}}{\Vert{}\mathbf{v}\Vert{}^2}\right) \mathbf{v}$$



**Why this perspective matters in AI:**

When an AI model uses an embedding (like a Word2Vec vector or a transformer token), a single concept (like "sentiment" or "gender") is often an arbitrary diagonal direction through a 768-dimensional space. To measure how much "sentiment" a word vector has, the model projects that vector onto the learned "sentiment direction."


#### NumPy Implementation


In [ ]:
import numpy as np

# 1. Define two arbitrary vectors (v is a tilted axis at 45 degrees)
u = np.array([2.0, 5.0])
v = np.array([3.0, 3.0])

# Step 1: Make v a unit direction vector (length = 1.0)
norm_v = np.linalg.norm(v)
unit_v = v / norm_v

# Step 2: Scalar projection = how far u goes along unit_v
scalar_projection = np.dot(u, unit_v)

# Step 3: Vector projection = coordinate * unit_v
vector_projection = scalar_projection * unit_v

# Verification: The remainder (residual) must be perpendicular to v
perpendicular_part = u - vector_projection
dot_check = np.dot(perpendicular_part, v)

print(f"Target direction v:         {v}")
print(f"Unit direction v_hat:       {unit_v}")
print(f"Coordinate along v (scalar): {scalar_projection:.4f}")
print(f"Projected vector:           {vector_projection}")
print(f"Perpendicular remainder:    {perpendicular_part}")
print(f"Dot product check (near 0): {dot_check:.10f}")

### Concept 3: Vector Norms ($L_1$, $L_2$, $L_\infty$)

#### Mathematical Explanation

A **norm** is a function that assigns a strictly positive length or size to a vector. In machine learning, norms measure the size of errors, weights, and feature representations.

All common vector norms belong to the family of **$L_p$ norms**, defined as:

$$\Vert{}\mathbf{x}\Vert{}_p = \left( \sum_{i=1}^{n} \vert{}x_i\vert{}^p \right)^{\frac{1}{p}}$$

The three most critical norms used in AI are:


**1. $L_1$ Norm (Taxicab / Manhattan Norm): $p = 1$**

The sum of absolute values:

$$\Vert{}\mathbf{x}\Vert{}_1 = \sum_{i=1}^{n} \vert{}x_i\vert{} = \vert{}x_1\vert{} + \vert{}x_2\vert{} + \dots + \vert{}x_n\vert{}$$

* **Intuition:** Walking on a grid of city blocks where you can only move horizontally and vertically.
* **Why it matters in AI:** Used in **Lasso regularization (L1 regularization)**. Because its contours have sharp corners on coordinate axes, it forces non-informative weights to become exactly zero, yielding **sparse models** (automatic feature selection).


**2. $L_2$ Norm (Euclidean Norm): $p = 2$**

The straight-line ruler distance from the origin:

$$\Vert{}\mathbf{x}\Vert{}_2 = \sqrt{\sum_{i=1}^{n} x_i^2} = \sqrt{x_1^2 + x_2^2 + \dots + x_n^2}$$

* **Intuition:** The direct hypotenuse distance (Pythagorean theorem generalized to $n$ dimensions).
* **Why it matters in AI:** Used in **Ridge regularization / Weight Decay (L2 regularization)**. It penalizes large weights heavily without forcing them to exact zeros, preventing overfitting and keeping models stable.


**3. $L_\infty$ Norm (Max / Chebyshev Norm): $p \to \infty$**

The single largest absolute component in the vector:

$$\Vert{}\mathbf{x}\Vert{}_\infty = \max_{i} \vert{}x_i\vert{}$$

* **Intuition:** The worst-case coordinate displacement.
* **Why it matters in AI:** Crucial in **Adversarial Robustness (e.g., FGSM attacks)**, where you constrain an attacker so that no individual pixel changes by more than a tiny threshold $\epsilon$.


Let's throw away the scary math notation and break **Vector Norms** down like a puzzle.


### What is a "Norm"? (Simple English)

A **norm** simply means: **"How big is this vector?"** or **"What is its length?"**

Imagine a vector is just an arrow starting from the center $(0,0)$. A norm is just a specific rule (or ruler) for measuring how long that arrow is.

Depending on which ruler you pick, you get a different number. We have 3 famous rulers in AI:

1. **$L_1$ ruler** (The City Walker)
2. **$L_2$ ruler** (The Bird Flying Straight)
3. **$L_\infty$ ruler** (The Worst-Case Picker)

### 1. The $L_1$ Norm (Manhattan / City Blocks)

#### How the formula looks:

$$\Vert{}\mathbf{x}\Vert{}_1 = \sum_{i=1}^{n} \vert{}x_i\vert{}$$

#### How to read it out loud:

> *"The L-one norm of vector x equals the sum of the absolute values of every element $x_i$."*

#### Breakdown of every piece:

* **$\Vert{}\mathbf{x}\Vert{}_1$**: The two double bars $\Vert{}\dots\Vert{}$ mean *"calculate the length"*, and the tiny small number $_1$ at the bottom tells you *"use the L-1 rule"*.
* **$\sum_{i=1}^{n}$**: That symbol ($\Sigma$, Sigma) just means **"add everything together"**, starting from the first item ($i=1$) to the last item ($n$).
* **$\vert{}x_i\vert{}$**: The single vertical bars mean **absolute value** (strip away negative signs). If you have $-4$, it becomes $+4$.
* **$x_i$**: The individual number at slot $i$ inside the vector.

#### Simple Real-World Analogy:

Imagine you are walking in a city laid out on a grid. You cannot walk through buildings diagonally. To reach your destination, you must walk 3 blocks East, then 4 blocks North.

Your total distance walked is:

$\vert{}3\vert{} + \vert{}4\vert{} = 7$.


### 2. The $L_2$ Norm (Euclidean / Straight Line)

#### How the formula looks:

$$\Vert{}\mathbf{x}\Vert{}_2 = \sqrt{\sum_{i=1}^{n} x_i^2}$$

#### How to read it out loud:

> *"The L-two norm of vector x equals the square root of the sum of each element squared."*

#### Breakdown of every piece:

* **$\Vert{}\mathbf{x}\Vert{}_2$**: Length of vector $\mathbf{x}$ using the $L_2$ rule.
* **$x_i^2$**: Take each number in the vector and multiply it by itself ($3 \times 3 = 9$, $-4 \times -4 = 16$). Squaring naturally makes negative numbers positive.
* **$\sum$**: Add all those squared numbers together ($9 + 16 = 25$).
* **$\sqrt{\dots}$**: Take the square root of that sum ($\sqrt{25} = 5$).

#### Simple Real-World Analogy:

This is the straight-line distance a bird flies (the classic Pythagoras theorem: $a^2 + b^2 = c^2$).

If you go 3 right and 4 up, the direct distance from the start is:

$\sqrt{3^2 + 4^2} = \sqrt{9 + 16} = \sqrt{25} = 5$.


### 3. The $L_\infty$ Norm (Max / Extreme Value)

#### How the formula looks:

$$\Vert{}\mathbf{x}\Vert{}_\infty = \max_{i} \vert{}x_i\vert{}$$

#### How to read it out loud:

> *"The L-infinity norm of vector x is the maximum absolute value among all its elements."*

#### Breakdown of every piece:

* **$\Vert{}\mathbf{x}\Vert{}_\infty$**: Length of vector $\mathbf{x}$ using the infinity rule.
* **$\max_i$**: Look through all entries and pick the single biggest number.
* **$\vert{}x_i\vert{}$**: Ignore minus signs first, then find the biggest.

#### Simple Real-World Analogy:

Imagine a team of workers. One made a 3-dollar error, another made a 4-dollar error, and another made a 1-dollar error. What is the single biggest mistake? It is 4.


### Why AI Cares (In plain terms)

* **$L_1$ makes models compact:** When training a neural network, if you punish the network based on the $L_1$ rule, it forces useless connections to become **exactly 0**. It shrinks the model and deletes useless features.
* **$L_2$ keeps models balanced:** If you punish the network using the $L_2$ rule, it stops any single weight from becoming too huge, keeping predictions stable and smooth.
* **$L_\infty$ stops hackers:** If someone tries to trick an image classifier by secretly tweaking pixels, $L_\infty$ guarantees that *no single pixel* is changed by more than a tiny imperceptible amount.


#### NumPy Implementation


In [6]:
import numpy as np

# Define a sample error/weight vector with positive and negative values
x = np.array([3.0, -4.0, 1.0, -2.0])

# --- 1. L1 Norm (Manhattan) ---
# Formula: sum(|x_i|)
l1_manual = np.sum(np.abs(x))
l1_numpy = np.linalg.norm(x, ord=1)

# --- 2. L2 Norm (Euclidean) ---
# Formula: sqrt(sum(x_i^2))
l2_manual = np.sqrt(np.sum(x ** 2))
l2_numpy = np.linalg.norm(x, ord=2)  # ord=2 is the default for np.linalg.norm

# --- 3. L_infinity Norm (Chebyshev / Max) ---
# Formula: max(|x_i|)
linf_manual = np.max(np.abs(x))
linf_numpy = np.linalg.norm(x, ord=np.inf)

print(f"Vector x:       {x}\n")
print(f"L1 Norm   (Manual): {l1_manual}   | (NumPy): {l1_numpy}")
print(f"L2 Norm   (Manual): {l2_manual:.4f} | (NumPy): {l2_numpy:.4f}")
print(f"Linf Norm (Manual): {linf_manual}   | (NumPy): {linf_numpy}")


Vector x:       [ 3. -4.  1. -2.]

L1 Norm   (Manual): 10.0   | (NumPy): 10.0
L2 Norm   (Manual): 5.4772 | (NumPy): 5.4772
Linf Norm (Manual): 4.0   | (NumPy): 4.0


### Concept 4: Cosine Similarity

### What is Cosine Similarity?

Imagine two arrows starting from the same center point $(0,0)$.

**Cosine Similarity answers one question:**

> *"Are these two arrows pointing in the exact same direction, regardless of how long they are?"*

* It does **not** care about magnitude or size (a short arrow and a massive arrow can point in the exact same direction).
* It only cares about the **angle** between them.

The score is always between **$-1$ and $+1$**:

* **$+1.0$**: Pointing in the **exact same direction** (angle is $0^\circ$). Perfect similarity.
* **$0.0$**: Pointing at a **right angle** ($90^\circ$). Completely unrelated / independent.
* **$-1.0$**: Pointing in the **exact opposite direction** ($180^\circ$). Total opposites.


### The Formula

$$\text{Cosine Similarity} = \cos(\theta) = \frac{\mathbf{u} \cdot \mathbf{v}}{\Vert{}\mathbf{u}\Vert{}_2 \Vert{}\mathbf{v}\Vert{}_2}$$

#### How to read it out loud:

> *"Cosine theta equals the dot product of vector u and vector v, divided by the L2 norm of u times the L2 norm of v."*

#### Breakdown of every piece:

* **$\cos(\theta)$**: The trigonometric cosine of angle $\theta$ between the two arrows.
* **$\mathbf{u} \cdot \mathbf{v}$**: The **Dot Product** (from Concept 1). It multiplies matching elements and adds them up.
* **$\Vert{}\mathbf{u}\Vert{}_2$**: The **$L_2$ Norm** (from Concept 3) of vector $\mathbf{u}$ (its length).
* **$\Vert{}\mathbf{v}\Vert{}_2$**: The **$L_2$ Norm** of vector $\mathbf{v}$ (its length).
* **The division**: Dividing by their lengths cancels out their scale. This is literally just **normalizing both vectors first**, then taking their dot product!

$$\text{Cosine Similarity} = \left(\frac{\mathbf{u}}{\Vert{}\mathbf{u}\Vert{}_2}\right) \cdot \left(\frac{\mathbf{v}}{\Vert{}\mathbf{v}\Vert{}_2}\right) = \hat{\mathbf{u}} \cdot \hat{\mathbf{v}}$$


### Why AI Cares (In plain terms)

* **Vector Search / RAG:** When you ask a chatbot a question, your query is turned into an embedding vector. The system scans millions of document vectors and uses **cosine similarity** to retrieve the documents pointing in the same semantic direction.
* **Word Embeddings:** The vector for `"king"` and the vector for `"queen"` point in almost the same direction in vector space, so their cosine similarity is high (e.g., $0.85$).


### NumPy Implementation

Even though vector `v` is twice as big as `u`, the cosine similarity is **`1.0`** ($0^\circ$ angle) because direction is all that matters.

In [ ]:
import numpy as np

# 1. Define two vectors (for example, semantic embeddings of two sentences)
# u: "I love programming"
# v: "Coding is my passion"
u = np.array([1.0, 2.0, 3.0])
v = np.array([2.0, 4.0, 6.0])  # Exactly twice as long, but pointing SAME direction!

# ==========================================
# Method A: Formula step-by-step
# ==========================================
# Step 1: Calculate Dot Product (Top part of formula)
dot_product = np.dot(u, v)

# Step 2: Calculate L2 norms (lengths) of both vectors (Bottom part)
norm_u = np.linalg.norm(u)
norm_v = np.linalg.norm(v)

# Step 3: Divide dot product by the product of lengths
cos_sim_formula = dot_product / (norm_u * norm_v)


# ==========================================
# Method B: Normalize first, then dot product
# ==========================================
u_normalized = u / norm_u
v_normalized = v / norm_v
cos_sim_normalized = np.dot(u_normalized, v_normalized)


# Optional: Convert cosine similarity score to actual angle in degrees
angle_radians = np.arccos(np.clip(cos_sim_formula, -1.0, 1.0))
angle_degrees = np.degrees(angle_radians)

print(f"Vector u: {u}")
print(f"Vector v: {v}\n")
print(f"Dot product (raw scale)     : {dot_product}")
print(f"Cosine Similarity (Formula) : {cos_sim_formula:.4f}")
print(f"Cosine Similarity (Normed)  : {cos_sim_normalized:.4f}")
print(f"Angle between vectors       : {angle_degrees:.1f} degrees")

### Concept 5: Matrix-Vector Multiplication

### What is Matrix-Vector Multiplication? 

Imagine an input vector is a list of raw ingredients:


$$\mathbf{x} = [\text{sugar}, \text{flour}, \text{eggs}]$$

A **matrix** is just a recipe book where **each row is a separate recipe**:

* Row 1: Recipe for **Cookies** (weights for sugar, flour, eggs)
* Row 2: Recipe for **Cake** (weights for sugar, flour, eggs)

When you multiply the **Recipe Matrix** by the **Ingredients Vector**, you run the **Dot Product** for every recipe row against your ingredients.

The output is a new vector:


$$\mathbf{y} = [\text{total cookies made}, \text{total cake made}]$$

> **Core Idea:** Matrix-vector multiplication is simply **multiple dot products stacked on top of each other**.

### The Formula

If a matrix $W$ has $m$ rows and $n$ columns, and vector $\mathbf{x}$ has length $n$:

$$\mathbf{y} = W\mathbf{x}$$

For each row $i$ of the output:


$$y_i = \sum_{j=1}^{n} W_{i,j} x_j = W_{i,1} x_1 + W_{i,2} x_2 + \dots + W_{i,n} x_n$$

#### How to read it out loud:

> *"The output vector y equals matrix W multiplied by vector x. Each element $y_i$ is the dot product of the $i$-th row of W with vector x."*

#### Breakdown of every piece:

* **$W$**: The weight matrix (shape: $\text{rows} \times \text{cols}$).
* **$\mathbf{x}$**: The input vector (must have length equal to the number of columns in $W$).
* **$W_{i,j}$**: The weight in row $i$ and column $j$.
* **$\mathbf{y}$**: The resulting vector (its length equals the number of rows in $W$).


### Why AI Cares (In plain terms)

This is the exact math of a **Linear Layer / Dense Layer** in neural networks:

$$\mathbf{y} = W\mathbf{x} + \mathbf{b}$$

1. The input $\mathbf{x}$ (e.g., a token embedding of size 4096) enters the layer.
2. The weight matrix $W$ transforms it into a new space (e.g., project from 4096 dimensions to 11008 dimensions).
3. The model adds a bias vector $\mathbf{b}$ (a baseline offset).

Every transformer feed-forward block, attention projection ($Q, K, V$), and classification head is built on this exact operation.


### NumPy Implementation


In [7]:
import numpy as np

# 1. Define an input vector x (3 features: e.g., age, income, education level)
x = np.array([2.0, 3.0, 1.0])

# 2. Define a weight matrix W (2 neurons/outputs, each looking at all 3 features)
# Shape: (2 rows, 3 columns)
W = np.array([
    [0.5,  1.0, -0.5],   # Neuron 1 weights (Row 0)
    [2.0, -1.0,  0.0]    # Neuron 2 weights (Row 1)
])

# ==========================================
# Method A: Manual Dot Product row-by-row
# ==========================================
# Row 0 dot x: (0.5*2.0) + (1.0*3.0) + (-0.5*1.0) = 1.0 + 3.0 - 0.5 = 3.5
# Row 1 dot x: (2.0*2.0) + (-1.0*3.0) + (0.0*1.0) = 4.0 - 3.0 + 0.0 = 1.0
row_0_output = np.dot(W[0], x)
row_1_output = np.dot(W[1], x)
y_manual = np.array([row_0_output, row_1_output])

# ==========================================
# Method B: Built-in np.dot / @ operator
# ==========================================
y_dot = np.dot(W, x)
y_matmul = W @ x

print("Input vector x shape :", x.shape)
print("Weight matrix W shape:", W.shape)
print("Output vector y shape:", y_matmul.shape)
print("\nManual row-by-row dot products:", y_manual)
print("NumPy W @ x result            :", y_matmul)

Input vector x shape : (3,)
Weight matrix W shape: (2, 3)
Output vector y shape: (2,)

Manual row-by-row dot products: [3.5 1. ]
NumPy W @ x result            : [3.5 1. ]
